# Intermediate 05: Matryoshka Dimensions

Every embedding you store costs disk, memory, and search time. A 1024-dimensional
index is 2.7x the size of a 384-dimensional one over the same corpus.

The obvious way to spend less is to pick a smaller model. **Matryoshka
representation learning** offers another: train a model so that the *first* N
components of its vector are themselves a usable embedding. You keep one model,
and choose the width at indexing time.

Ollama exposes this through the `dimensions` parameter on `embed`. This notebook
measures what you actually give up.

**The honest question this asks:** truncation is only free if quality holds. So
we will not assume — we will retrieve against ground truth at several widths and
look at the numbers.

**You will need:** `foundation/02` run at least once, so there is a corpus.

In [ ]:
# --- ragkit: shared utilities ---
from ragkit import config, db, registry
from ragkit.db import table_name_for
from ragkit.embed import embed_texts, embed_one
from ragkit.metrics import ndcg_at_k, precision_at_k, recall_at_k
from ragkit.retrieval import cosine_similarity

import time

print(config.describe())
conn = db.connect()

## What truncation actually does

A Matryoshka model is trained so that its 768-dimensional output degrades
*gracefully* when you keep only the first 256 components. That is not true of an
ordinary embedding model — slicing an arbitrary vector throws away arbitrary
information.

First, confirm the API does what we think.

In [ ]:
WIDTHS = [128, 256, 512, 768]
PROBE = "The sky appears blue because shorter wavelengths scatter more."

for width in WIDTHS:
    vec = embed_one(PROBE, dimensions=width)
    print(f"dimensions={width:5d} -> len={len(vec):5d}  first 3: {[round(v, 4) for v in vec[:3]]}")

Notice the leading components are *not* identical across widths. Ollama
re-normalizes after truncating, so a 128-wide vector is the first 128 components
rescaled to unit length. That is what makes them comparable to each other — but
it also means you must never compare a 128-wide vector to a 768-wide one.

That mistake is easy to make and produces plausible-looking nonsense, which is
exactly why `ragkit.retrieval.cosine_similarity` raises on a dimension mismatch
instead of silently returning a number.

In [ ]:
try:
    cosine_similarity(embed_one(PROBE, dimensions=128), embed_one(PROBE, dimensions=768))
except ValueError as exc:
    print(f"Refused, correctly:\n  {exc}")

## Does meaning survive truncation?

A first check: at each width, does the model still rank a paraphrase above an
unrelated sentence? If the ordering ever flips, that width has lost the plot.

In [ ]:
QUERY      = "why is the sky blue"
PARAPHRASE = "The sky looks blue because short wavelengths scatter more in air."
UNRELATED  = "Chlorophyll absorbs light, which is why leaves appear green."

print(f"{'width':>6}  {'paraphrase':>11}  {'unrelated':>10}  {'margin':>8}  ordering")
for width in WIDTHS:
    q, p, u = (embed_one(t, dimensions=width) for t in (QUERY, PARAPHRASE, UNRELATED))
    sim_p, sim_u = cosine_similarity(q, p), cosine_similarity(q, u)
    ok = "correct" if sim_p > sim_u else "FLIPPED"
    print(f"{width:6d}  {sim_p:11.4f}  {sim_u:10.4f}  {sim_p - sim_u:8.4f}  {ok}")

Watch the **margin** column rather than the raw similarities. Absolute cosine
values drift with dimensionality and mean very little on their own; what matters
for retrieval is whether the right answer stays ahead of the wrong one, and by
how much. A shrinking margin means the ranking is getting fragile even while it
is still technically correct.

## The cost side

Storage scales linearly with width. Search time does too, for a brute-force scan;
with an HNSW index the relationship is softer but still real. Measure both.

In [ ]:
CORPUS = [
    "Photosynthesis converts light energy into chemical energy in plants.",
    "Chlorophyll absorbs red and blue light, reflecting green.",
    "The sky appears blue because short wavelengths scatter more.",
    "Rayleigh scattering explains the colour of the daytime sky.",
    "Mitochondria produce ATP, the energy currency of the cell.",
    "Photosynthesis occurs in the chloroplasts of plant cells.",
] * 40   # 240 chunks: enough for timings to mean something

print(f"{'width':>6}  {'embed (s)':>10}  {'floats':>9}  {'~KB':>7}  {'vs 768':>7}")
baseline = None
for width in WIDTHS:
    start = time.time()
    vectors = embed_texts(CORPUS, dimensions=width)
    elapsed = time.time() - start

    floats = len(vectors) * width
    kb = floats * 4 / 1024          # pgvector stores float4
    baseline = baseline or kb
    print(f"{width:6d}  {elapsed:10.2f}  {floats:9,}  {kb:7.0f}  {kb / baseline:6.2f}x")

## Retrieval quality against ground truth

Similarity margins are suggestive; retrieval metrics are the real test. Build a
small labelled set and measure NDCG@3 at each width.

Note this uses the **corrected** NDCG. An earlier version of this project
normalised against the best ordering *of what was retrieved*, which made the
metric blind to recall — finding one relevant chunk out of ten and ranking it
first scored a perfect 1.000. See `src/ragkit/metrics.py`.

In [ ]:
DOCS = [
    "Photosynthesis converts light energy into chemical energy in plants.",
    "Chlorophyll absorbs red and blue light, reflecting green.",
    "Photosynthesis occurs in the chloroplasts of plant cells.",
    "The sky appears blue because short wavelengths scatter more.",
    "Rayleigh scattering explains the colour of the daytime sky.",
    "Sunsets look red because light travels through more atmosphere.",
    "Mitochondria produce ATP, the energy currency of the cell.",
    "Ribosomes assemble proteins from amino acids.",
]

# question -> indices into DOCS that genuinely answer it
GROUND_TRUTH = {
    "how do plants make energy":   [0, 2],
    "why is the sky blue":         [3, 4],
    "what powers a cell":          [6],
    "what makes leaves green":     [1],
}

print(f"{'width':>6}  {'NDCG@3':>8}  {'P@3':>7}  {'R@3':>7}")
for width in WIDTHS:
    doc_vecs = embed_texts(DOCS, dimensions=width)
    scores = []
    for question, relevant in GROUND_TRUTH.items():
        qv = embed_one(question, dimensions=width)
        ranked = sorted(range(len(DOCS)),
                        key=lambda i: cosine_similarity(qv, doc_vecs[i]),
                        reverse=True)
        scores.append((
            ndcg_at_k(ranked, relevant, k=3),
            precision_at_k(ranked, relevant, k=3),
            recall_at_k(ranked, relevant, k=3),
        ))
    n, p, r = (sum(c) / len(c) for c in zip(*scores))
    print(f"{width:6d}  {n:8.3f}  {p:7.3f}  {r:7.3f}")

## Reading the result

Put the two tables side by side. The question is not "which width scores best" —
full width almost always does. It is **where the quality curve bends relative to
the cost curve.**

Storage falls linearly: 256 dimensions is a third the size of 768. Quality
usually does not fall linearly; it holds, then drops off. The useful width is the
last one before that drop.

Two caveats worth carrying away:

1. **This corpus is tiny.** Four questions over eight documents is a
   demonstration, not evidence. Differences of a few points here are noise. Run
   it against the real ground truth in `evaluation-lab/01` before believing any
   specific number.
2. **Truncation is a property of the model, not of embeddings in general.**
   `nomic-embed-text` is trained for it. Slicing a model that is not will degrade
   far faster, and nothing in the API will warn you.

## Storing a truncated model

If you decide a narrower width is worth it, register it as its own catalog entry
rather than quietly truncating at write time. The registry records the dimension,
the DDL reads it from there, and `registry.resolve()` verifies the table actually
matches — so a width change can never silently mix with existing vectors.

Add it to `EMBEDDING_MODELS` in `src/ragkit/models.py`, then run
`python scripts/preflight.py` to confirm the model really returns what the
catalog claims.

In [ ]:
for model in registry.list_models(conn):
    print(f"  {model.alias:20s} dim={model.dimension:5d}  rows={model.embedding_count:6,}  {model.table_name}")

conn.close()